In [27]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

In [28]:
df = pd.read_csv("train.csv")

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [30]:
df.drop(columns=["PassengerId","Name","Ticket","Cabin"], inplace= True)

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


In [32]:
df["Age"] = df.groupby(["Pclass","Sex"])["Age"].transform(lambda x: x.fillna(x.median()))

df["Fare"] = df["Fare"].fillna(df["Fare"].median())

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       891 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


In [34]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
embarked_mapping = {"S": 0, "C": 1, "Q": 2}
df["Embarked"] = df["Embarked"].map(embarked_mapping)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [35]:
X = df.drop(columns= ["Survived"])
y = df["Survived"]

In [36]:
from sklearn.model_selection import train_test_split

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)

In [38]:
from sklearn.preprocessing import StandardScaler

In [39]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [40]:
X_train_t = torch.tensor(X_train_scaled, dtype= torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype= torch.float32)

y_train_t = torch.tensor(y_train.values, dtype= torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test.values, dtype= torch.float32).unsqueeze(1)

In [41]:
from torch import nn

In [42]:
class TitanicClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer = nn.Sequential(
            nn.Linear(7,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.layer(x)

In [43]:
model = TitanicClassifier()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr= 0.001)

In [44]:
def calculate_accuracy(y_true, y_pred):
    correct = torch.eq(y_true.squeeze(), y_pred.squeeze()).sum().item()
    return (correct / len(y_true)) * 100

In [45]:
epochs = 200

for epoch in range(epochs):

    model.train()
    logits = model(X_train_t).squeeze()
    loss = loss_fn(logits, y_train_t.squeeze())
    preds = torch.round(logits)
    acc = calculate_accuracy(y_train_t, preds)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.inference_mode():
        test_logits = model(X_test_t).squeeze()
        test_loss = loss_fn(test_logits, y_test_t.squeeze())
        test_preds = torch.round(test_logits)
        test_acc = calculate_accuracy(y_test_t, test_preds)

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | "
              f"Train Loss: {loss:.4f}, Train Acc: {acc:.2f}% | "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

Epoch   0 | Train Loss: 0.6811, Train Acc: 61.52% | Test Loss: 0.6737, Test Acc: 61.45%
Epoch  20 | Train Loss: 0.6113, Train Acc: 74.44% | Test Loss: 0.6144, Test Acc: 75.98%
Epoch  40 | Train Loss: 0.5169, Train Acc: 80.06% | Test Loss: 0.5347, Test Acc: 78.77%
Epoch  60 | Train Loss: 0.4556, Train Acc: 81.18% | Test Loss: 0.4689, Test Acc: 79.89%
Epoch  80 | Train Loss: 0.4350, Train Acc: 81.74% | Test Loss: 0.4461, Test Acc: 80.45%
Epoch 100 | Train Loss: 0.4144, Train Acc: 82.87% | Test Loss: 0.4403, Test Acc: 80.45%
Epoch 120 | Train Loss: 0.4139, Train Acc: 81.46% | Test Loss: 0.4376, Test Acc: 81.01%
Epoch 140 | Train Loss: 0.3893, Train Acc: 82.72% | Test Loss: 0.4371, Test Acc: 79.89%
Epoch 160 | Train Loss: 0.3968, Train Acc: 83.85% | Test Loss: 0.4362, Test Acc: 79.33%
Epoch 180 | Train Loss: 0.4088, Train Acc: 84.13% | Test Loss: 0.4342, Test Acc: 79.89%


In [46]:
from pathlib import Path

In [47]:
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents= True, exist_ok= True)

MODEL_NAME = "titanic.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

torch.save(model.state_dict(), f = MODEL_SAVE_PATH)

In [48]:
loaded_model = TitanicClassifier()

In [49]:
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))

<All keys matched successfully>

In [50]:
# .\venv\Scripts\python.exe -m uvicorn main:app --reload --port 8000 bunu terminale yazarsan çalışır